In [1]:
import mne
import os
import numpy as np
from os import walk
from itertools import groupby
import itertools

In [2]:
root = ('experiments/')
list_file_names = []
for path, dirs, files in os.walk(root):
    for file in files:
        if file.endswith(".bdf"):
             list_file_names.append(os.path.join(root, file))
print(len(list_file_names),'files')
files = []
path = 'experiments/'
participant_prefix = list_file_names[0][12:-4]
for (dirpath, dirnames, filenames) in walk(path):
    new_names = [dirpath+f for f in filenames if (participant_prefix in f)]
    files.extend(new_names)
    break

21 files


In [3]:
raw = mne.io.read_raw_bdf(files[0],preload=True)
events = mne.find_events(raw, stim_channel=None,initial_event=False)
excluded = ['EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8']
raw.info['bads'] = excluded
raw.pick_types(meg=False, eeg=True, eog=False, exclude='bads')
data = raw._data
sfreq = raw.info['sfreq']
ch_names = raw.info['ch_names']

Extracting EDF parameters from F:\PhD\materials_neuroscience\EEG_work\trials\experiments\aq values deu neg.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 831487  =      0.000 ...   406.000 secs...
Trigger channel Status has a non-zero initial value of {initial_value} (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
245 events found on stim channel Status
Event IDs: [  1   2   4   8   9  10  16  25  32  64  73 128]
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


In [4]:
data.shape

(32, 831488)

In [5]:
test_list = [item[2] for item in events]

In [6]:
patterns = [i+[item] for item in [16,32,64,128] for i in [[item]+[1,2,4,8] for item in [9,10]]]

In [7]:
patterns

[[9, 1, 2, 4, 8, 16],
 [10, 1, 2, 4, 8, 16],
 [9, 1, 2, 4, 8, 32],
 [10, 1, 2, 4, 8, 32],
 [9, 1, 2, 4, 8, 64],
 [10, 1, 2, 4, 8, 64],
 [9, 1, 2, 4, 8, 128],
 [10, 1, 2, 4, 8, 128]]

In [8]:
curr_group = []
pat_len = len(patterns[0])
counter = 0

#for i, x in enumerate(test_list):
#    if counter == i*pat_len:
#        print([test_list[x-pat_len],test_list[x-1],test_list[x]],counter)
#        counter +=pat_len

#for i in range(len(test_list)):
#    if counter == i*pat_len:
#        print(test_list[counter-1],counter)
#        if counter < len(test_list)-pat_len:
#            counter +=pat_len
#        else:
#            break

epoch_list = []
for i in range(len(test_list)):
    if counter == i*pat_len:
        if [test_list[item] for item in range(counter-pat_len,counter)] in patterns:
            epoch_list.append([test_list[item] for item in range(counter-pat_len,counter)])
        #else:
        #    counter = counter-5
    if counter < len(test_list)-pat_len:
        counter +=pat_len
    else:
        break
epoch_list

[[9, 1, 2, 4, 8, 128], [10, 1, 2, 4, 8, 32], [9, 1, 2, 4, 8, 64]]

In [9]:
# most updated
patterns = [i+[item] for item in [16,32,64,128] for i in [[item]+[1,2,4,8] for item in [9,10]]]
pat_len = len(patterns[0])
epoch_list = []
counter = 0
for i in range(len(test_list)):
    if test_list[i] in [16,32,64,128]:
        for item in range(i,0,-1):
            if test_list[item] in [9,10]:
                epoch_list.append(test_list[item:i+1])
                break
            if i-item >= pat_len-1:
                epoch_list.append(test_list[item-1:i+1])
                break
    if counter < len(test_list):
        counter+=1
    else:
        break
print(epoch_list)

[[10, 1, 2, 4, 16], [10, 1, 2, 4, 16, 8, 16], [10, 1, 2, 4, 8, 64], [8, 64, 1, 2, 4, 8, 128], [8, 128, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 64], [10, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 32], [10, 1, 2, 4, 8, 32], [9, 1, 2, 4, 8, 64], [10, 1, 2, 4, 8, 16], [16, 25, 1, 2, 4, 8, 16], [9, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 64], [9, 1, 2, 4, 16], [9, 1, 2, 4, 16, 8, 16], [9, 1, 2, 4, 8, 128], [10, 1, 2, 4, 8, 32], [9, 1, 2, 4, 8, 64], [8, 64, 1, 2, 4, 8, 16], [9, 1, 2, 4, 8, 16], [9, 1, 2, 4, 16], [9, 1, 2, 4, 16, 8, 16], [8, 16, 1, 2, 4, 8, 64], [64, 73, 1, 2, 4, 8, 32], [10, 1, 2, 4, 8, 64], [10, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 64], [10, 1, 2, 4, 8, 32], [10, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 16], [8, 16, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 16], [9, 1, 2, 4, 8, 32], [9, 1, 2, 4, 8, 32], [10, 1, 2, 4, 8, 128], [9, 1, 2, 4, 8, 128], [9, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 64], [10, 1, 2, 4, 8, 64, 128]]


In [10]:
l = []
for item in events:
    l.append(item[2])
print(l)

[128, 1, 2, 4, 8, 64, 10, 1, 2, 4, 16, 8, 16, 10, 1, 2, 4, 8, 64, 1, 2, 4, 8, 128, 1, 2, 4, 8, 16, 10, 1, 2, 4, 8, 16, 10, 1, 2, 4, 8, 64, 10, 1, 2, 4, 8, 64, 9, 1, 2, 4, 8, 64, 9, 1, 2, 4, 8, 32, 10, 1, 2, 4, 8, 32, 9, 1, 2, 4, 8, 64, 10, 1, 2, 4, 8, 16, 25, 1, 2, 4, 8, 16, 9, 1, 2, 4, 8, 16, 10, 1, 2, 4, 8, 64, 9, 1, 2, 4, 16, 8, 16, 9, 1, 2, 4, 8, 128, 10, 1, 2, 4, 8, 32, 9, 1, 2, 4, 8, 64, 1, 2, 4, 8, 16, 9, 1, 2, 4, 8, 16, 9, 1, 2, 4, 16, 8, 16, 1, 2, 4, 8, 64, 73, 1, 2, 4, 8, 32, 10, 1, 2, 4, 8, 64, 10, 1, 2, 4, 8, 16, 10, 1, 2, 4, 8, 64, 9, 1, 2, 4, 8, 64, 10, 1, 2, 4, 8, 32, 10, 1, 2, 4, 8, 64, 9, 1, 2, 4, 8, 64, 9, 1, 2, 4, 8, 16, 1, 2, 4, 8, 16, 10, 1, 2, 4, 8, 16, 9, 1, 2, 4, 8, 32, 9, 1, 2, 4, 8, 32, 10, 1, 2, 4, 8, 128, 9, 1, 2, 4, 8, 128, 9, 1, 2, 4, 8, 16, 10, 1, 2, 4, 8, 64, 128]


In [132]:
epoch_list = []
counter = 0
for i in range(len(test_list)):
    if counter >= 0:
        if test_list[i] in [16,32,64,128]:
            for item in range(i,0,-1):
                if test_list[item] in [9,10]:
                    epoch_list.append(test_list[item:i+1])
                    break
    else:
        continue
    if counter < len(test_list):
        counter+=1
    else:
        break
print(epoch_list)

[[10, 1, 2, 4, 16], [10, 1, 2, 4, 16, 8, 16], [10, 1, 2, 4, 8, 64], [10, 1, 2, 4, 8, 64, 1, 2, 4, 8, 128], [10, 1, 2, 4, 8, 64, 1, 2, 4, 8, 128, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 64], [10, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 32], [10, 1, 2, 4, 8, 32], [9, 1, 2, 4, 8, 64], [10, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 16, 25, 1, 2, 4, 8, 16], [9, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 64], [9, 1, 2, 4, 16], [9, 1, 2, 4, 16, 8, 16], [9, 1, 2, 4, 8, 128], [10, 1, 2, 4, 8, 32], [9, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 64, 1, 2, 4, 8, 16], [9, 1, 2, 4, 8, 16], [9, 1, 2, 4, 16], [9, 1, 2, 4, 16, 8, 16], [9, 1, 2, 4, 16, 8, 16, 1, 2, 4, 8, 64], [9, 1, 2, 4, 16, 8, 16, 1, 2, 4, 8, 64, 73, 1, 2, 4, 8, 32], [10, 1, 2, 4, 8, 64], [10, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 64], [10, 1, 2, 4, 8, 32], [10, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 64], [9, 1, 2, 4, 8, 16], [9, 1, 2, 4, 8, 16, 1, 2, 4, 8, 16], [10, 1, 2, 4, 8, 16], [9, 1, 2, 4, 8, 32], [9, 1, 2, 4, 8, 

In [51]:
len(l)

245

In [115]:
res = []
curr_group = []
for i, x in enumerate(test_list):
    if i == 0 and x not in [9,10]:
        # start a new group
        curr_group = ['f']
        res.append(curr_group)
    elif i == 0 or x in [9,10]:
        # start a new group
        curr_group = [x]
        res.append(curr_group)
    if x in [1,2,4,8]:
        # add to current group
        curr_group.append(x)
        i=+1
        x=+1
    if x in [16,32,64,128]:
        curr_group.append(x)
    #else:
    #    curr_group.append('f')
res

[['f', 128, 1, 2, 4, 8, 64],
 [10, 1, 2, 4, 16, 8, 16],
 [10, 1, 2, 4, 8, 64, 1, 2, 4, 8, 128, 1, 2, 4, 8, 16],
 [10, 1, 2, 4, 8, 16],
 [10, 1, 2, 4, 8, 64],
 [10, 1, 2, 4, 8, 64],
 [9, 1, 2, 4, 8, 64],
 [9, 1, 2, 4, 8, 32],
 [10, 1, 2, 4, 8, 32],
 [9, 1, 2, 4, 8, 64],
 [10, 1, 2, 4, 8, 16, 1, 2, 4, 8, 16],
 [9, 1, 2, 4, 8, 16],
 [10, 1, 2, 4, 8, 64],
 [9, 1, 2, 4, 16, 8, 16],
 [9, 1, 2, 4, 8, 128],
 [10, 1, 2, 4, 8, 32],
 [9, 1, 2, 4, 8, 64, 1, 2, 4, 8, 16],
 [9, 1, 2, 4, 8, 16],
 [9, 1, 2, 4, 16, 8, 16, 1, 2, 4, 8, 64, 1, 2, 4, 8, 32],
 [10, 1, 2, 4, 8, 64],
 [10, 1, 2, 4, 8, 16],
 [10, 1, 2, 4, 8, 64],
 [9, 1, 2, 4, 8, 64],
 [10, 1, 2, 4, 8, 32],
 [10, 1, 2, 4, 8, 64],
 [9, 1, 2, 4, 8, 64],
 [9, 1, 2, 4, 8, 16, 1, 2, 4, 8, 16],
 [10, 1, 2, 4, 8, 16],
 [9, 1, 2, 4, 8, 32],
 [9, 1, 2, 4, 8, 32],
 [10, 1, 2, 4, 8, 128],
 [9, 1, 2, 4, 8, 128],
 [9, 1, 2, 4, 8, 16],
 [10, 1, 2, 4, 8, 64, 128]]

In [ ]:
for item in test_list:
    
head = test_list[0]
tail = test_list[1:]
[group] + split_consecutive_groups([x for x in tail if x != head])

In [160]:
def groups(lst):
    if not lst:
        return []
    head = lst[0]
    tail = lst[1:]
    group = [head] + [x for x in tail if x in [1,2,4,8]]
    return [group] + groups([x for x in tail if x in [1,2,4,8]])

In [161]:
groups(test_list) 
#x if x in [1,2,4,8] else return for x in tail
#x for x in tail if x in [1,2,4,8]

[[128,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8],
 [1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  4,
  8,
  1,
  2,
  

In [ ]:
participant_data = np.ndarray((len(files),n_channels,n_times))

for trial in range(0,len(files)):
    new_data = np.loadtxt(files[trial], delimiter=',')
    if trial == 0:
        print('n_channels, n_times: ' + str(new_data.shape))
    new_data = new_data.astype(float)
    participant_data[trial] = new_data

print('Number of epochs: ' + str(participant_data.shape))

In [ ]:
epochs_info = mne.create_info(ch_names, sfreq, ch_types='eeg')
epochs = mne.EpochsArray(data=participant_data, info=epochs_info, events=epoch_events_num, event_id=event_id)

In [37]:
np.ndarray((2,3,4))

array([[[-0.25, -0.5 ,  0.  , -0.25],
        [ 0.25, -0.5 ,  0.5 , -0.25],
        [ 0.25,  0.  ,  0.5 ,  0.25]],

       [[ 0.25,  0.5 ,  0.  ,  0.25],
        [-0.25,  0.5 , -0.5 ,  0.25],
        [-0.25,  0.  , -0.5 , -0.25]]])